# 04 · Evaluation, export and push to the Hugging Face Hub

**Inputs:** outputs of 01, 02 and (optionally) 03. **Secret:** `HF_TOKEN` (write access). Computes per-language perplexity and bits-per-byte, compares against GPT-2 small on identical text (bits-per-byte, since per-token perplexity is not comparable across tokenizers), checks language-ID of generations, writes charts, exports safetensors and pushes base weights to the repo root and SFT weights to `sft/`.

In [ ]:
# --- Get the Trilingua code -------------------------------------------------------
# Option A (after the repo is public): clone it.
# Option B: upload the repo folder as a Kaggle Dataset named "trilingua-code" and attach it.
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/aladewealth/Trilingual-.git"
CODE = "/tmp/trilingua"
attached = glob.glob("/kaggle/input/**/trilingua/model.py", recursive=True)
if os.path.exists(CODE):
    pass
elif attached:
    shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), CODE)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE); sys.path.insert(0, CODE)
print("code at", CODE, os.listdir(CODE))

In [ ]:
# Optional secrets (Add-ons -> Secrets). Nothing breaks if they are missing.
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
for k in ("WANDB_API_KEY", "HF_TOKEN"):
    v = get_secret(k)
    if v:
        os.environ[k] = v
        print(k, "loaded")

In [ ]:
# Locate the data produced by 01_tokenizer_and_data (attach that notebook's output as input)
metas = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert metas, "Attach the output of 01_tokenizer_and_data as an input dataset"
DATA = os.path.dirname(metas[0])
print("DATA =", DATA); print(open(os.path.join(DATA, "meta.json")).read())
from trilingua.utils import find_latest_checkpoint
BASE = find_latest_checkpoint(['/kaggle/input'])
SFT = find_latest_checkpoint(['/kaggle/input'], name='sft_final.pt')
TRAIN_DIR = os.path.dirname(BASE)
print('base', BASE, '\nsft', SFT)
!pip -q install fasttext-wheel 2>&1 | tail -1   # optional; falls back to a heuristic lang-id
HF_REPO = 'aladewealth/trilingua'   # TODO: placeholder repo id
RES = '/kaggle/working/results'

In [ ]:
!python -m trilingua.evaluate --model {BASE} --data_dir {DATA} --out_dir {RES} --compare gpt2 \
    --train_dir {TRAIN_DIR} --tag base --langid_n 50 --device cuda

In [ ]:
if SFT:
    !python -m trilingua.evaluate --model {SFT} --data_dir {DATA} --out_dir {RES} --chat --tag sft --langid_n 30 --device cuda

In [ ]:
from IPython.display import Image, Markdown
for f in sorted(glob.glob(f'{RES}/*.png')): print(f); display(Image(f))
display(Markdown(open(f'{RES}/samples_base.md').read()))

## Export to safetensors and push

In [ ]:
!python -m trilingua.export --ckpt {BASE} --tokenizer {DATA}/tokenizer.json --out_dir /kaggle/working/export
if SFT:
    !python -m trilingua.export --ckpt {SFT} --tokenizer {DATA}/tokenizer.json --out_dir /kaggle/working/export_sft
if os.environ.get('HF_TOKEN'):
    !python -m trilingua.export --ckpt {BASE} --tokenizer {DATA}/tokenizer.json --out_dir /kaggle/working/export --push {HF_REPO}
    if SFT:
        !python -m trilingua.export --ckpt {SFT} --tokenizer {DATA}/tokenizer.json --out_dir /kaggle/working/export_sft --push {HF_REPO} --path_in_repo sft
else:
    print('HF_TOKEN secret not set: skipping push')

In [ ]:
# CPU sanity check of the exported model (what the Gradio app does)
!python generate.py --model /kaggle/working/export --lang fr --prompt "Il était une fois" --max_new_tokens 60 --seed 0
if SFT:
    !python generate.py --model /kaggle/working/export_sft --lang es --mode write --prompt "los volcanes" --max_new_tokens 80 --seed 0